In [5]:
import sys
sys.path.append('../')

import torch
from transformers import RobertaTokenizerFast
from src.model import RobertaForQA
from src.utils import RobertaConfig
from safetensors.torch import load_file

In [6]:
class InferenceModel:
    def __init__(self, path_to_weights=None, hf_model=True):
        if hf_model:
            self.config = RobertaConfig(pretrained_backbone='pretrained_huggingface')
        else:
            self.config = RobertaConfig(pretrained_backbone='random')

        self.tokenizer = RobertaTokenizerFast.from_pretrained(self.config.hf_model_name)

        self.model = RobertaForQA(self.config)
        if path_to_weights is not None:
            weights = load_file(path_to_weights)
            self.model.load_state_dict(weights)
        self.model.eval()

    def inference(self, question, context):
        inputs = self.tokenizer(
            text=question,
            text_pair=context,
            max_length=self.config.context_length,
            truncation='only_second',
            return_tensors='pt'
        )

        with torch.no_grad():
            start_logits, end_logits = self.model(**inputs)

        start_token_idx = start_logits.squeeze().argmax().item()
        end_token_idx = end_logits.squeeze().argmax().item()

        tokens = inputs['input_ids'].squeeze()[start_token_idx:end_token_idx+1]
        answer = self.tokenizer.decode(tokens, skip_special_tokens=True).strip()

        return start_token_idx, end_token_idx, answer

In [7]:
context = "Water boils at 100 degrees Celsius at sea level, but the boiling point decreases at higher altitudes."
question = "At what temperature does water boil at sea level?"

context2 = """
        The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. 
        It is named after the engineer Gustave Eiffel, whose company designed and built the tower. 
        Constructed from 1887 to 1889 as the entrance to the 1889 World's Fair, it was initially criticized 
        by some of France's leading artists and intellectuals for its design, but it has become a global 
        cultural icon of France."""
question2 = "When was the Eiffel Tower constructed?"

context3 = """
        Neural networks are a subset of machine learning and are at the heart of deep learning algorithms. 
        Their name and structure are inspired by the human brain, mimicking the way that biological neurons 
        signal to one another. A typical neural network is composed of layers of nodes, much like the neurons 
        of the human brain."""
question3 = "What inspired the structure of neural networks?"

inferencer = InferenceModel(path_to_weights='../work_dir/LoRA_RoBERTa_QA/checkpoints/checkpoint_merged_2.safetensors',
                            hf_model=True)
start_token_idx, end_token_idx, answer = inferencer.inference(question, context)
print(f'{start_token_idx}-{end_token_idx}: {answer}')
start_token_idx, end_token_idx, answer = inferencer.inference(question2, context2)
print(f'{start_token_idx}-{end_token_idx}: {answer}')
start_token_idx, end_token_idx, answer = inferencer.inference(question3, context3)
print(f'{start_token_idx}-{end_token_idx}: {answer}')

Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FileNotFoundError: No such file or directory: "work_dir/LoRA_RoBERTa_QA/checkpoints/checkpoint_merged_2.safetensors"